# Image Super-Resolution with Neural Fields

Recover a high-resolution (HR) image from its 4× bicubic downsampling, the
degradation of the DIV2K benchmark. No model sees the HR image: each renders HR
pixels, which are downsampled the way the benchmark made the low-resolution
(LR) image and fitted to it. Following the standard protocol, PSNR and SSIM are
computed on the luma (Y) channel after shaving a 4-pixel border, and LPIPS on
RGB.

## Setup

In [ ]:
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor

import neurofield as nf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Load Images

In [ ]:
SCALE = 4
HR_PATH = "../data/DIV2K/DIV2K_valid_HR/0882.png"
LR_PATH = f"../data/DIV2K/DIV2K_valid_LR_bicubic/X{SCALE}/0882x{SCALE}.png"


image_hr, image_lr = (Image.open(path).convert("RGB") for path in (HR_PATH, LR_PATH))
original, observed = pil_to_tensor(image_hr), pil_to_tensor(image_lr)  # uint8 (C, H, W)
(H, W), (h, w) = original.shape[1:], observed.shape[1:]
print(f"HR size: {H} x {W}, LR size: {h} x {w}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, image, title in zip(
    axes, (image_hr, image_lr), ("High resolution", f"Low resolution ({SCALE}×)")
):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.show()

## Forward Model

Antialiased bicubic downsampling, which reproduces the benchmark's MATLAB
`imresize`, is linear and separable: one matrix per axis describes it exactly,
borders included. Resizing the rows of an identity image writes that matrix
out. Each of its rows has at most `4 * SCALE` nonzero taps, the footprint of
one LR pixel along the axis.

In [ ]:
def resize_matrix(size_in, size_out, mode="bicubic", antialias=True):
    """The ``(size_out, size_in)`` matrix of ``F.interpolate`` along one axis."""
    identity = torch.eye(size_in, device=device)[None, None]
    size = (size_out, size_in)
    return F.interpolate(identity, size=size, mode=mode, antialias=antialias)[0, 0]


def downsample(image, kernels):
    """Apply one kernel per axis: ``(..., H, W, C)`` to ``(..., h, w, C)``."""
    return torch.einsum("...il,...lmc,...jm->...ijc", kernels[0], image, kernels[1])


kernels = [resize_matrix(H, h), resize_matrix(W, w)]  # (h, H) and (w, W)

# The forward model applied to the HR image gives the benchmark's LR image.
simulated = downsample(original.movedim(0, -1).float().to(device), kernels)
simulated = simulated.round().clamp(0, 255).movedim(-1, 0).cpu()
fidelity = nf.psnr(simulated, observed, data_range=255)
print(f"Taps per LR pixel and axis: {int((kernels[0] != 0).sum(1).max())}")
print(f"Forward model against the LR image: {fidelity:.2f} dB")

## Minibatches

`SuperResolutionDataset` draws random blocks of LR pixels that cover a fraction
`subsample` of the LR image, and returns the HR coordinates under their
kernels, the kernel rows, and the LR pixels. Neighbouring LR pixels share most
of their footprint, so a block needs far fewer field evaluations per pixel than
single pixels do: 30 in an 8 × 8 block against 256 alone.

The loss renders each footprint, downsamples it along both axes, and takes the
mean absolute error (L1) against the LR pixels, the usual loss in
super-resolution.

In [ ]:
class SuperResolutionDataset(nf.ImageCoordinateDataset):
    """LR pixels as targets, the HR coordinates under their kernels as inputs.

    Items hold ``input``, the HR coordinates of each block's footprint, of
    shape ``(B, L, L, 2)``; ``weight_y`` and ``weight_x``, the kernel rows of
    each block on its footprint, of shape ``(B, block, L)``; and ``target``,
    the LR pixels in ``[-1, 1]``, of shape ``(B, block, block, C)``.
    """

    def __init__(self, image, kernels, subsample=1.0, block=8):
        device = kernels[0].device
        self.original = self.load(image, mode="RGB")  # uint8 (C, h, w)
        self.id = Path(image).stem
        self.kernels, self.subsample, self.block = kernels, subsample, block
        self.grid_size = tuple(kernel.shape[1] for kernel in kernels)  # (H, W)
        self.input = self.create_input().to(device)  # HR coordinates, (H, W, 2)
        self.target = self.preprocess(self.original).to(device)  # LR pixels, (h, w, C)

        # The first tap of each kernel row, and the footprint of a block along one axis.
        self.first = [(kernel != 0).int().argmax(1) for kernel in kernels]
        taps = max(int((kernel != 0).sum(1).max()) for kernel in kernels)
        scale = self.grid_size[0] // self.target.shape[0]
        self.length = scale * (block - 1) + taps

    def __getitem__(self, index):
        device = self.target.device
        h, w = self.target.shape[:2]
        count = max(1, round(self.subsample * h * w / self.block**2))
        offsets = torch.arange(self.block, device=device)
        span = torch.arange(self.length, device=device)

        pixels, taps, weights = [], [], []
        for size, full, kernel, first in zip(
            (h, w), self.grid_size, self.kernels, self.first
        ):
            start = torch.randint(size - self.block + 1, (count, 1), device=device)
            pixel = start + offsets  # (B, b)
            tap = first[start].clamp(max=full - self.length) + span  # (B, L)
            weights.append(kernel[pixel[:, :, None], tap[:, None, :]])  # (B, b, L)
            pixels.append(pixel)
            taps.append(tap)

        return {
            "id": self.id,
            "input": self.input[taps[0][:, :, None], taps[1][:, None, :]],
            "weight_y": weights[0],
            "weight_x": weights[1],
            "target": self.target[pixels[0][:, :, None], pixels[1][:, None, :]],
            "_original": self.original,
        }


def l1_loss(output, target, peak=2.0):
    """Mean absolute error, with the PSNR of the fit for the training log."""
    loss = F.l1_loss(output, target)
    mse = F.mse_loss(output, target).item()
    return loss, {"l1": loss.item(), "psnr": 10 * math.log10(peak**2 / mse)}


def field_loss(batch, model):
    """Render each footprint, downsample it along both axes, and match the LR pixels."""
    weights = batch["weight_y"], batch["weight_x"]
    return l1_loss(downsample(model(batch["input"]), weights), batch["target"])

## Model Configurations

Every model has about 500k parameters. Each baseline takes the smallest size
of its kind that reaches this budget, and FUTON the largest CP rank that stays
within it, so FUTON is never the larger model. The budget is close to the 519k
values of the LR image, so each model can fit what it observes and the
comparison rests on what it adds between the pixels.

FUTON's configuration follows from the task: one basis function per LR pixel
along each axis (a sinc or a Lanczos kernel), the bandwidth that the LR image
determines, and its usual decoder, a one-layer MLP with a tanh output. The
baselines are those of the image-representation notebook, widened to the
budget.

Every learning rate, and the setting that governs each baseline's scale
(marked `# tuned`), maximizes the mean PSNR over three DIV2K training images
(0395, 0431 and 0777) under the protocol of this notebook.

In [ ]:
BUDGET = 500_000  # parameters


def futon_rank(budget):
    """The largest CP rank at which FUTON has at most ``budget`` parameters.

    Rank ``R`` costs ``R * (h + w)`` factor entries, ``R * R + R`` for the
    hidden layer and ``3 * R + 3`` for the output layer.
    """
    b = h + w + 4
    return int((math.sqrt(b * b + 4 * (budget - 3)) - b) / 2)


CONFIGS = [
    {
        "name": "RFF",  # Tancik et al. (NeurIPS 2020), 2D image regression
        "class": nf.RFF,
        "kwargs": {
            "hidden_features": 387,
            "hidden_layers": 3,
            "num_frequencies": 256,
            "sigma": 10.0,  # tuned
            "output_activation": torch.tanh,
        },
        "lr": 3e-3,
    },
    {
        "name": "PE-MLP",  # FINER image benchmark
        "class": nf.PEMLP,
        "kwargs": {
            "hidden_features": 401,
            "hidden_layers": 4,
            "num_frequencies": 10,  # tuned
            "output_activation": torch.tanh,
        },
        "lr": 3e-4,
    },
    {
        "name": "MFN",  # GaborNet (Fathony et al., ICLR 2021)
        "class": nf.MFN,
        "kwargs": {
            "hidden_features": 404,
            "hidden_layers": 4,
            "input_scale": 256.0,  # tuned
            "alpha": 1.25,
            "beta": 0.125,
            "weight_scale": 1.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "SIREN",  # FINER image benchmark
        "class": nf.SIREN,
        "kwargs": {
            "hidden_features": 407,
            "hidden_layers": 4,
            "omega": 120.0,  # tuned
            "output_activation": torch.tanh,
        },
        "lr": 1e-4,
    },
    {
        "name": "Gauss",  # FINER image benchmark
        "class": nf.Gauss,
        "kwargs": {
            "hidden_features": 407,
            "hidden_layers": 4,
            "scale": 30.0,  # tuned
            "output_activation": torch.tanh,
        },
        "lr": 3e-4,
    },
    {
        "name": "WIRE",  # FINER image benchmark, real Gabor layer
        "class": nf.RealWIRE,
        "kwargs": {
            "hidden_features": 408,
            "hidden_layers": 4,
            "omega": 10.0,  # tuned
            "scale": 10.0,  # tuned
            "output_activation": torch.tanh,
        },
        "lr": 3e-3,
    },
    {
        "name": "FINER",  # FINER image benchmark
        "class": nf.FINER,
        "kwargs": {
            "hidden_features": 407,
            "hidden_layers": 4,
            "omega": 120.0,  # tuned
            "output_activation": torch.tanh,
        },
        "lr": 1e-4,
    },
    {
        "name": "Instant-NGP",  # instant-ngp configs/image/base.json
        "class": nf.InstantNGP,
        "kwargs": {
            "num_levels": 17,
            "features_per_level": 2,
            "log2_hashmap_size": 15,
            "base_resolution": 16,
            "max_resolution": max(h, w),  # tuned
            "hidden_features": 64,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 3e-3,
        "train_kwargs": {"adam_betas": (0.9, 0.99), "adam_eps": 1e-15},
    },
    {
        "name": "TensoRF",
        "class": nf.TensoRF,
        "kwargs": {
            "rank": 566,
            "resolution": max(h, w) // 2,  # tuned
            "mode": "cp",
            "hidden_features": 256,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
    {
        "name": "GA-Planes",
        "class": nf.GAPlanes,
        "kwargs": {
            "features": 170,
            "resolution": (max(h, w), 32),  # tuned
            "hidden_features": 256,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "FUTON-sinc",
        "class": nf.FUTON,
        "kwargs": {
            "basis": ("sinc", {"num_components": (h, w), "grid_size": (H, W)}),
            "combiner": ("cp", {"rank": futon_rank(BUDGET)}),
            "decoder": ("mlp", {"hidden_layers": 1}),
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "FUTON-lanczos",
        "class": nf.FUTON,
        "kwargs": {
            "basis": ("lanczos", {"num_components": (h, w), "radius": 3}),
            "combiner": ("cp", {"rank": futon_rank(BUDGET)}),
            "decoder": ("mlp", {"hidden_layers": 1}),
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
]

In [ ]:
sizes = pd.Series(
    {
        config["name"]: nf.count_parameters(
            config["class"](in_features=2, out_features=3, **config["kwargs"])
        )
        for config in CONFIGS
    }
)
is_futon = sizes.index.str.startswith("FUTON")

assert len(sizes) == len(CONFIGS), "model names must be unique"
assert sizes[~is_futon].min() >= BUDGET >= sizes[is_futon].max()
print(sizes.to_string())

## Train Neural Fields

Each step draws 8 × 8 blocks of LR pixels that cover 5% of the LR image,
renders the HR pixels under their kernels, downsamples them, and compares them
with the LR pixels. Every model trains for 2000 steps from the same seed, with
Adam and a cosine-annealed learning rate. The HR image serves only to score
the render every 100 steps, and the reported result is the last one.

In [ ]:
SUBSAMPLE, BLOCK = 0.05, 8  # share of the LR pixels per step, and the side of a block
TRAIN = {"num_epochs": 2000, "log_interval": 100, "eval_interval": 100}


def shave(image):
    """Move an image to the device, without its SCALE-pixel border."""
    return image.to(device)[:, SCALE:-SCALE, SCALE:-SCALE]


def luma(image):
    """BT.601 luma of a shaved uint8 RGB image, as MATLAB's rgb2ycbcr computes it."""
    weights = torch.tensor([65.481, 128.553, 24.966], device=device) / 255
    return 16 + torch.einsum("chw,c->hw", shave(image).float(), weights)[None]


METRICS = {
    "psnr": lambda pred, target: nf.psnr(luma(pred), luma(target), data_range=255),
    "ssim": lambda pred, target: nf.ssim(luma(pred), luma(target), data_range=255),
    "lpips": lambda pred, target: nf.lpips(shave(pred), shave(target)),
}

train_dataset = SuperResolutionDataset(
    LR_PATH, kernels, subsample=SUBSAMPLE, block=BLOCK
)
eval_dataset = nf.ImageCoordinateDataset(HR_PATH)
results = []

for config in CONFIGS:
    torch.manual_seed(0)

    model = config["class"](in_features=2, out_features=3, **config["kwargs"])

    print(f"\n--- {config['name']} ({sizes[config['name']]:,} params) ---")
    res = nf.train(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss_fn=field_loss,
        lr=config["lr"],
        metrics=METRICS,
        chunk_size=2**18,
        device=device,
        **TRAIN,
        **config.get("train_kwargs", {}),
    )

    output = nf.chunked_inference(
        model, eval_dataset.input, chunk_size=2**18, device=device
    )
    res["config"]["model"] = config["name"]
    res["reconstructed"] = eval_dataset.postprocess(output.cpu())
    results.append(res)

    del model, output, res["model_dict"]
    torch.cuda.empty_cache()

## Train DIP Baseline

Deep Image Prior (Ulyanov et al., CVPR 2018) renders the whole HR image from
fixed noise with a convolutional network of the same size, fitted through the
same forward model and loss for as many steps.

In [ ]:
def prior_loss(batch, model):
    """Render the HR image from noise, downsample it, and match the LR image."""
    output = downsample(model(batch["input"]).movedim(1, -1), kernels)
    return l1_loss(output, batch["target"].movedim(1, -1), peak=1.0)


torch.manual_seed(0)

dip_eval_dataset = nf.DIPImageDataset(HR_PATH, noise_channels=32, reg_noise_std=0)
# The same seeded noise, at the HR size, perturbed at each step.
dip_train_dataset = nf.DIPImageDataset(
    LR_PATH, noise_shape=dip_eval_dataset.input.shape
)

# 60 channels per scale instead of 128 meet the budget.
model = nf.DIPSkip(in_channels=32, down_channels=[60] * 5, up_channels=[60] * 5)

print(f"\n--- DIP ({nf.count_parameters(model):,} params) ---")
res = nf.train(
    model=model,
    train_dataset=dip_train_dataset,
    eval_dataset=dip_eval_dataset,
    loss_fn=prior_loss,
    lr=3e-3,  # tuned
    metrics=METRICS,
    device=device,
    **TRAIN,
)

output = nf.chunked_inference(model, dip_eval_dataset.input[None], device=device)
res["config"]["model"] = "DIP"
res["reconstructed"] = dip_eval_dataset.postprocess(output[0].cpu())
results.append(res)

del model, output, res["model_dict"]
torch.cuda.empty_cache()

## Interpolation Baselines

PIL's bicubic filter is that of MATLAB's `imresize`, the standard baseline.

In [ ]:
LABELS = {"psnr": "PSNR (dB)", "ssim": "SSIM", "lpips": "LPIPS"}

interpolations, interpolation_records = {}, []

for name in ("Nearest", "Bilinear", "Bicubic"):
    start = time.perf_counter()
    upsampled = image_lr.resize((W, H), Image.Resampling[name.upper()])
    duration = time.perf_counter() - start

    interpolations[name] = pil_to_tensor(upsampled)
    interpolation_records.append(
        {
            "Model": name,
            "# Params (k)": 0.0,
            "Iteration": 0,
            "Time (s)": 0.0,
            "Speed (img/s)": 1 / duration,
            **{
                label: METRICS[metric](interpolations[name], original)
                for metric, label in LABELS.items()
            },
        }
    )

## Collect Results

In [ ]:
records = []
for res in results:
    for entry in res["history"]:
        if "eval" not in entry:
            continue

        records.append(
            {
                "Model": res["config"]["model"],
                "# Params (k)": res["config"]["num_params"] / 1000,
                "Iteration": entry["epoch"],
                "Time (s)": entry["elapsed"],
                "Speed (img/s)": 1.0 / entry["eval"]["duration"],
                **{label: entry["eval"][metric] for metric, label in LABELS.items()},
            }
        )


records = pd.DataFrame(records)
interpolation_records = pd.DataFrame(interpolation_records)

## Convergence Performance

The dotted line is bicubic interpolation.

In [ ]:
bicubic = interpolation_records.set_index("Model").loc["Bicubic", "PSNR (dB)"]
floor = min(records.groupby("Model")["PSNR (dB)"].last().min(), bicubic - 6)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True, layout="constrained")

for ax, x in zip(axes, ("Iteration", "Time (s)")):
    sns.lineplot(
        data=records,
        x=x,
        y="PSNR (dB)",
        hue="Model",
        style="Model",
        markers=True,
        legend=ax is axes[1],
        ax=ax,
    )
    ax.axhline(bicubic, color="0.4", linestyle=":", label="Bicubic")
    ax.set_ylim(floor - 0.5, bicubic + 3)
    ax.grid(alpha=0.3)

axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.show()

## Results Summary

In [ ]:
summary = (
    pd.concat([records, interpolation_records], ignore_index=True)
    .groupby("Model")
    .agg(
        {
            "# Params (k)": "first",
            "Iteration": "last",
            "Time (s)": "last",
            "Speed (img/s)": "mean",
            "PSNR (dB)": "last",
            "SSIM": "last",
            "LPIPS": "last",
        }
    )
    .sort_values("PSNR (dB)", ascending=False)
    .reset_index()
)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(36, 9))

for ax, metric in zip(axes, ["PSNR (dB)", "SSIM", "LPIPS", "Speed (img/s)"]):
    summary_sorted = summary.sort_values(metric, ascending=True)
    ax.barh(summary_sorted["Model"], summary_sorted[metric])
    ax.set_xlabel(metric)
    ax.grid(axis="x", alpha=0.3)

axes[-1].set_xscale("log")  # interpolation is orders of magnitude faster
plt.show()

## Visual Comparison

A zoom on the butterfly's head: the antennae are thinner than an LR pixel, so
what each model draws there comes from its inductive bias.

In [ ]:
ZOOM = (slice(380, 636), slice(960, 1216))  # rows, columns of the HR image

psnr_lookup = summary.set_index("Model")["PSNR (dB)"]
outputs = {res["config"]["model"]: res["reconstructed"] for res in results}
outputs |= interpolations
ranked = sorted(outputs, key=psnr_lookup.get, reverse=True)

items = [("Original", original)] + [(name, outputs[name]) for name in ranked]
num_cols = math.ceil(len(items) / 3)
fig, axes = plt.subplots(
    3, num_cols, figsize=(2.6 * num_cols, 9.6), layout="constrained"
)

for ax, (name, image) in zip(axes.flat, items):
    ax.imshow(image[:, ZOOM[0], ZOOM[1]].permute(1, 2, 0))
    title = name if name == "Original" else f"{name}\n{psnr_lookup[name]:.2f} dB"
    ax.set_title(title)
for ax in axes.flat:
    ax.axis("off")

plt.show()